# 01 -- Build dataset (Step 3: SFT data pipeline; Step 13: the scaled 500k set)

Step 3 (done on Colab, Sep 28; the labeled dataset was copied to the RunPod volume in Step 6): download one month of real Lichess games -> sample positions by game phase -> label each with
Stockfish -> save as Parquet under `artifact_root`. This is the data SFT (Step 5) trains on.

**Gate:**
1. The labeled dataset is saved.
2. The phase mix matches `configs/data.yaml`.
3. The Stockfish CPU time spent labeling is logged (it counts against the SFT compute budget).

**Step 13 (plan v5), `BUILD = "scaled"`:** the same pipeline at 500k positions, sampled fresh with the same rules,
reading `2013-06` and then later months until full. It skips the 50k set's validation games and positions, so every
dataset size shares that one validation set, and it reports how many positions are near-wins or simple endings.
Run it on a CPU-only pod from its own repo clone (`docs/runpod_runbook.md`, section G); the notebook uses the clone it
is run from.

**Run it twice.** First with `TIMING_RUN=1` set in the shell (5,000 positions) to measure labeling speed on this machine's CPU, then
without it for the full dataset. Each run saves to its own folder.

**Safe to interrupt.** Labels are saved in chunks. If the session drops, run the notebook again from the top:
finished chunks are kept and only the rest gets labeled. CPU only -- no GPU runtime needed.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
# Before each session, in a pod terminal (code is never edited on the pod):
#   cd /workspace/Chess_Bot && git fetch && git checkout BRANCH && git reset --hard origin/BRANCH
import os
# The clone this notebook was opened from (its notebooks/ folder's parent), so a second clone,
# e.g. /workspace/Chess_Bot_data on a labeling pod, runs its own code (plan v5 Sec 4.1).
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import yaml
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
RAW = os.path.join(ROOT, 'data', 'raw')              # the downloaded Lichess month
PROCESSED = os.path.join(ROOT, 'data', 'processed')  # labeled datasets
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters

Everything comes from `configs/data.yaml` except `BUILD` and `TIMING_RUN`. Each dataset goes to its own folder, named
after the month, label depth, and size, so the timing run and the full run never mix.

In [ ]:
import json
import time

import chess
import yaml

import pandas as pd

from data.annotate import annotate, load_labels
from data.dataset import PositionDataset, fixed_split, load_datasets, near_win_report, split_by_game
from data.download import ensure_month, iter_months
from data.sample_positions import PHASES, PhaseRule, phase_quotas, position_key, sample_positions

BUILD = "scaled"    # "base": the Step 3 50k set (done Sep 28). "scaled": the Step 13 500k set (plan v5).
# TIMING_RUN: 5,000 positions first, to measure speed (and, for scaled, see the near-win report early); off = the
# full size. Set per run from the pod shell (`TIMING_RUN=1 jupyter nbconvert ...`, runbook G), so the notebook
# needs no commit between the two runs. Default: off.
TIMING_RUN = os.environ.get("TIMING_RUN", "0") == "1"

cfg = yaml.safe_load(open("configs/data.yaml"))
if BUILD not in ("base", "scaled"):
    raise ValueError(f"BUILD must be 'base' or 'scaled', got {BUILD!r}")
SCALED = BUILD == "scaled"
MONTHS = cfg["scaled"]["source_months"] if SCALED else [cfg["source_month"]]
FULL = cfg["scaled"]["num_positions"] if SCALED else cfg["num_positions"]
NUM_POSITIONS = 5000 if TIMING_RUN else FULL
PHASE_MIX = cfg["phase_mix"]
RULE = PhaseRule(**cfg["phase_rule"])
DEPTH = cfg["stockfish_label_depth"]
WORKERS = os.cpu_count()   # one single-threaded Stockfish per core
NAME = f"sft_scaled_d{DEPTH}_n{NUM_POSITIONS}" if SCALED else f"sft_{MONTHS[0]}_d{DEPTH}_n{NUM_POSITIONS}"
OUT_DIR = os.path.join(PROCESSED, NAME)

# The fixed validation set (scaled only): the 50k set's by-game validation split. Its games and
# positions are kept out of the new sample, so no validation position is ever trained on.
BASE = None
EXCLUDE_GAMES, EXCLUDE_KEYS = set(), set()
if SCALED:
    BASE = load_labels(os.path.join(PROCESSED, cfg["scaled"]["val_from"]))
    _, val_df = split_by_game(BASE, cfg["val_fraction"], cfg["seed"])
    EXCLUDE_GAMES = set(val_df["game_id"])
    EXCLUDE_KEYS = {position_key(chess.Board(f)) for f in val_df["fen"]}
    print(f"fixed validation set: {len(val_df)} positions from {len(EXCLUDE_GAMES)} games "
          f"({cfg['scaled']['val_from']}), kept out of the sample")

print(f"months {MONTHS} | {NUM_POSITIONS} positions, targets {phase_quotas(NUM_POSITIONS, PHASE_MIX)}")
print(f"phase rule {RULE} | label depth {DEPTH} | {WORKERS} workers")
print("output:", OUT_DIR)

## Download
Rated games, cached in `data/raw/` and checked against Lichess's published checksum. Only the first run
downloads; later runs reuse the file. For `scaled`, months after the first are downloaded during sampling, only if
reached.

In [ ]:
path = ensure_month(MONTHS[0], RAW)
print(f"{path.name}: {path.stat().st_size / 1e6:.0f} MB, checksum matches Lichess's published one")


## Sample positions
Reads games in file order (month by month) and keeps at most one position per phase per game, skipping positions already taken,
until each phase's quota is full. Same months + same seed = same sample every run. The sample is saved
(`positions.parquet`), so a rerun after an interruption reuses it instead of reading the games again.

In [ ]:
POSITIONS = os.path.join(OUT_DIR, "positions.parquet")
STATS = os.path.join(OUT_DIR, "sampling_stats.json")
os.makedirs(OUT_DIR, exist_ok=True)
if os.path.exists(POSITIONS):
    records = pd.read_parquet(POSITIONS).to_dict("records")
    stats = json.load(open(STATS))
    print(f"reusing the saved sample: {len(records)} positions")
else:
    used = []   # filled by iter_months: which month files were read, with their checksums
    t0 = time.perf_counter()
    records, stats = sample_positions(iter_months(MONTHS, RAW, used=used), NUM_POSITIONS, PHASE_MIX, RULE,
                                      seed=cfg["seed"], exclude_games=EXCLUDE_GAMES, exclude_keys=EXCLUDE_KEYS)
    stats = {**stats, "months": used, "phase_rule": vars(RULE), "seed": cfg["seed"],
             "sampling_seconds": round(time.perf_counter() - t0)}
    with open(STATS, "w") as f:
        json.dump(stats, f, indent=2)
    if stats["complete"]:
        pd.DataFrame(records).to_parquet(POSITIONS, index=False)
print(f"read {stats['games_read']} games ({stats['games_excluded']} validation games skipped) from "
      f"{[m['month'] for m in stats['months']]} in {stats['sampling_seconds']}s; "
      f"skipped {stats['duplicates_skipped']} repeated positions")
print("targets:", stats["targets"])
print("counts: ", stats["counts"])
if not stats["complete"]:
    raise RuntimeError("SHORT: the listed months ran out of games -- add months to configs/data.yaml")


## Label with Stockfish
Each position gets Stockfish's best move and a win probability for the side to move, at a fixed depth. The
Stockfish CPU time is recorded -- it's charged to the SFT budget. If this cell is interrupted, just rerun it.

In [ ]:
manifest = annotate(records, OUT_DIR, depth=DEPTH, workers=WORKERS, chunk_size=cfg["label_chunk_size"])
cpu = manifest["stockfish_cpu_seconds"]
print(f"\n{manifest['rows']} labeled positions | Stockfish CPU {cpu / 60:.1f} min ({cpu / manifest['rows']:.3f} s/position)")
if manifest["labeled_this_run"]:
    wall_per = manifest["wall_seconds_this_run"] / manifest["labeled_this_run"]
    full = FULL
    print(f"this run: {manifest['wall_seconds_this_run'] / 60:.1f} min wall")
    print(f"projected full dataset ({full}): {full * cpu / manifest['rows'] / 3600:.1f} CPU hours, "
          f"~{full * wall_per / 3600:.1f} hours wall on this machine")

## Gate check

In [ ]:
labels = load_labels(OUT_DIR)
manifest = json.load(open(os.path.join(OUT_DIR, "manifest.json")))
targets = phase_quotas(NUM_POSITIONS, PHASE_MIX)
counts = {p: int((labels["phase"] == p).sum()) for p in PHASES}
legal = all(chess.Move.from_uci(m) in chess.Board(f).legal_moves for f, m in zip(labels["fen"], labels["best_move"]))
keys = labels["fen"].map(lambda f: position_key(chess.Board(f)))
if SCALED:   # all of it trains; validation is the fixed 50k split, none of whose games or positions may appear here
    train_df, val_df = fixed_split(labels, BASE, cfg["val_fraction"], cfg["seed"])
    no_leak = len(train_df) == len(labels) and not keys.isin(EXCLUDE_KEYS).any()
else:
    train_df, val_df = split_by_game(labels, cfg["val_fraction"], cfg["seed"])
    no_leak = True

checks = {
    f"labeled dataset saved ({len(labels)} positions)":
        os.path.exists(os.path.join(OUT_DIR, "labels.parquet")) and len(labels) == NUM_POSITIONS,
    f"phase mix matches config {counts}": counts == targets,
    f"labeling CPU time logged ({manifest['stockfish_cpu_seconds'] / 60:.1f} min)": manifest["stockfish_cpu_seconds"] > 0,
    "every label move is legal; win probabilities in [0, 1]": legal and labels["win_prob"].between(0, 1).all(),
    "no repeated positions": keys.is_unique,
    f"train / validation share no game ({len(train_df)} / {len(val_df)} positions)":
        set(train_df["game_id"]).isdisjoint(set(val_df["game_id"])),
    "no validation game or position in the sample (scaled)": no_leak,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
GATE = all(checks.values())
print("\nGATE", "PASSED" if GATE else "NOT PASSED -- look at the failing line")

# Near-win check (plan v5 Step 13, option A): one side close to winning (win prob > 0.9 or < 0.1), simple endings
# (<= 6 pieces), and positions from games that ended in checkmate -- per phase. Enrichment only if there is a gap.
print("\n", near_win_report(labels).to_string())
if "ending" in labels:
    print("\ngame endings (share of positions):", labels["ending"].value_counts(normalize=True).round(3).to_dict())

## What training will see
Plain records -- the board as text, Stockfish's move, the win probability. The prompt around them is built in
Step 4 (`model/llm_policy.py`), so training and play read exactly the same text.

In [ ]:
if SCALED:
    train, val = PositionDataset(train_df, "structured"), PositionDataset(val_df, "structured")
else:
    train, val = load_datasets(os.path.join(OUT_DIR, "labels.parquet"), "structured", cfg["val_fraction"], cfg["seed"])
print(f"train {len(train)} | validation {len(val)}\n")
item = train[0]
print(item["board_text"])
print(f"\nlabel move: {item['label_move']} | win prob (side to move): {item['label_value']:.3f} | phase: {item['phase']}")
print("\nsame position as FEN:", item["fen"])